In [18]:
from algorithms.actor_critic.actor_critic_discrete import CartPoleEnvWrapper,ACConfig,train_actor_critic,infer_action_dim, to_numpy_state,DiscreteActorCriticAgent

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm
from itertools import product

%load_ext autoreload
%autoreload 2




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
env = CartPoleEnvWrapper()

state = env.reset()
print("Reset state:", state)
print("State shape:", state.shape)
print("Action space:", env.action_space)

next_s, r, d, _ = env.step(0)
print("Step output:", next_s, r, d)

Reset state: [-0.04058227  0.04756223  0.02611397  0.02860643]
State shape: (4,)
Action space: [0, 1]
Step output: [-0.03963102 -0.14792429  0.0266861   0.32941288] 1.0 False


In [11]:
gammas = [0.99, 0.995]
configs = []
seed_base = 100

num_episodes = 2500
max_steps_per_episode = 500

# MODIFIED: Better learning rates for CartPole
learning_rates = [2e-3, 3e-3, 5e-3]  # Added 2e-3, removed 7e-3

# MODIFIED: Add 0.005 for lower entropy
entropy_coefs = [0.02, 0.01, 0.005]  # Added 0.005

# MODIFIED: Added 256, removed 64 (too small)
hidden_sizes = [128, 256]  # 64 is too small for CartPole

for i, lr in enumerate(learning_rates):
    for j, g in enumerate(gammas):
        for k, ent in enumerate(entropy_coefs):
            for m, h in enumerate(hidden_sizes):
                label = f"lr={lr}_gamma={g}_ent={ent}_h={h}"
                cfg = ACConfig(
                    num_episodes=num_episodes,
                    max_steps_per_episode=max_steps_per_episode,
                    gamma=g,
                    lr=lr,
                    entropy_coef=ent,
                    hidden_size=h,
                    seed=seed_base + 100*i + 10*j + k + m,
                    label=label,
                )
                configs.append(cfg)

print("Total configs:", len(configs))
print("First 10 configs:")
for cfg in configs[:10]:
    print(f"  {cfg.label}")

Total configs: 36
First 10 configs:
  lr=0.002_gamma=0.99_ent=0.02_h=128
  lr=0.002_gamma=0.99_ent=0.02_h=256
  lr=0.002_gamma=0.99_ent=0.01_h=128
  lr=0.002_gamma=0.99_ent=0.01_h=256
  lr=0.002_gamma=0.99_ent=0.005_h=128
  lr=0.002_gamma=0.99_ent=0.005_h=256
  lr=0.002_gamma=0.995_ent=0.02_h=128
  lr=0.002_gamma=0.995_ent=0.02_h=256
  lr=0.002_gamma=0.995_ent=0.01_h=128
  lr=0.002_gamma=0.995_ent=0.01_h=256


In [12]:
cart_results = []

for cfg in configs:
    print(f"\n=== Running Config: {cfg.label} ===")
    env = CartPoleEnvWrapper()
    result = train_actor_critic(env, cfg, log_interval=250)
    cart_results.append(result)



=== Running Config: lr=0.002_gamma=0.99_ent=0.02_h=128 ===
Episode 250/2500, Return: 12.00, Best: 94.00, LR: 0.001982, Entropy: 0.0176
Episode 500/2500, Return: 30.00, Best: 108.00, LR: 0.001876, Entropy: 0.0156
Episode 750/2500, Return: 29.00, Best: 108.00, LR: 0.001685, Entropy: 0.0137
Episode 1000/2500, Return: 15.00, Best: 108.00, LR: 0.001426, Entropy: 0.0121
Episode 1250/2500, Return: 22.00, Best: 108.00, LR: 0.001125, Entropy: 0.0107
Episode 1500/2500, Return: 14.00, Best: 108.00, LR: 0.000813, Entropy: 0.0094
Episode 1750/2500, Return: 13.00, Best: 108.00, LR: 0.000518, Entropy: 0.0083
Episode 2000/2500, Return: 22.00, Best: 108.00, LR: 0.000271, Entropy: 0.0074
Episode 2250/2500, Return: 17.00, Best: 108.00, LR: 0.000095, Entropy: 0.0065
Episode 2500/2500, Return: 43.00, Best: 108.00, LR: 0.000008, Entropy: 0.0057

=== Running Config: lr=0.002_gamma=0.99_ent=0.02_h=256 ===
Episode 250/2500, Return: 14.00, Best: 95.00, LR: 0.001982, Entropy: 0.0176
Episode 500/2500, Return: 12

In [13]:
best_result = max(cart_results, key=lambda r: r["best_reward"])
best_cfg = best_result["config"]

print("\n🔥 BEST CONFIG FOUND 🔥")
print(f"Label: {best_cfg.label}")
print(f"Learning Rate: {best_cfg.lr}")
print(f"Hidden Size: {best_cfg.hidden_size}")
print(f"Best Reward Achieved: {best_result['best_reward']:.1f}")




🔥 BEST CONFIG FOUND 🔥
Label: lr=0.005_gamma=0.99_ent=0.02_h=128
Learning Rate: 0.005
Hidden Size: 128
Best Reward Achieved: 173.0
